In [3]:
%load_ext autoreload
%autoreload 2

import torch
import numpy as np
import os
import sys
sys.path.append('/home/matanyaw/DIP_decoder/voxel_embeddings_ROIs')
import ROI_coverage
# Getting my modules
sys.path.append('/home/jonathak/VisualEncoder/Analysis/Brain_maps')
from NIPS_utils import get_hemisphere_indices, get_roi_indices, get_roi_indices_per_hemisphere


# Setting up GPU
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# Appending Roman's path
sys.path.append('/home/romanb/PycharmProjects/BrainVisualReconst/')

In [4]:
# Loading the model
encoder = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14_reg')
model = torch.load('/home/jonathak/VisualEncoder/Voxels_Prediction/model_ch128.pth').eval().cuda()

# Testing voxel embeddings
voxel_embeddings = model.voxel_embed # Has shape [315997, 256]

# Getting subject 1 indices

subject = 1

lh_start, lh_end = get_hemisphere_indices(subject, 'lh')
rh_start, rh_end = get_hemisphere_indices(subject, 'rh')    
sub_indices = np.arange(lh_start, rh_end)

voxel_embeddings = voxel_embeddings[sub_indices]


Using cache found in /home/matanyaw/.cache/torch/hub/facebookresearch_dinov2_main
/home/matanyaw/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/matanyaw/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/matanyaw/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


In [5]:
subject = 1

ROI_names = ROI_coverage.get_roi_names(subject=subject)

predefined_ROI_indices = {}

# Creating a dictionary of ROI indices (iterating over copy because we remove ROIs that don't exist)
for ROI in ROI_names.copy():
    
    roi_indices = get_roi_indices(subject, ROI)
    
    if roi_indices is None:
        ROI_names.remove(ROI)
    else:
        predefined_ROI_indices[ROI] = roi_indices
        # print(ROI, roi_indices.shape)
# predefined_ROIless_indeices = summary_roi_coverage(predefined_ROI_indices, sub_indices)


ROI 'mTL-bodies' not found for subject 1
ROI 'mTL-faces' not found for subject 1
ROI 'aTL-faces' not found for subject 1
ROI 'mTL-words' not found for subject 1


In [14]:
roi_coverage_configs = []

roi_coverage_configs.append(ROI_coverage.InferRoiCoverageConfig(voxel_embeddings=voxel_embeddings, predefined_ROI_indices_dict=predefined_ROI_indices,
                                                                center_method='mean', metric='euclidean', discrimination_method='predefined'))

roi_coverage_configs.append(ROI_coverage.InferRoiCoverageConfig(voxel_embeddings=voxel_embeddings, predefined_ROI_indices_dict=predefined_ROI_indices,
                                                                center_method='mean', metric='euclidean', discrimination_method='nearest_voxels'))

roi_coverage_configs.append(ROI_coverage.InferRoiCoverageConfig(voxel_embeddings=voxel_embeddings, predefined_ROI_indices_dict=predefined_ROI_indices,
                                                                center_method='mean', metric='cosine', discrimination_method='nearest_voxels'))

roi_coverage_configs.append(ROI_coverage.InferRoiCoverageConfig(voxel_embeddings=voxel_embeddings, predefined_ROI_indices_dict=predefined_ROI_indices,
                                                                center_method='meanshift', metric='euclidean', discrimination_method='nearest_voxels'))


roi_coverage_configs.append(ROI_coverage.InferRoiCoverageConfig(voxel_embeddings=voxel_embeddings, predefined_ROI_indices_dict=predefined_ROI_indices,
                                                                center_method='meanshift', metric='cosine', discrimination_method='nearest_voxels'))


roi_coverage_configs.append(ROI_coverage.InferRoiCoverageConfig(voxel_embeddings=voxel_embeddings, predefined_ROI_indices_dict=predefined_ROI_indices,
                                                                center_method='meanshift', metric='euclidean', discrimination_method='nearest_center'))

roi_coverage_configs.append(ROI_coverage.InferRoiCoverageConfig(voxel_embeddings=voxel_embeddings, predefined_ROI_indices_dict=predefined_ROI_indices,
                                                                center_method='meanshift', metric='cosine', discrimination_method='nearest_center'))

In [16]:
for coverage in roi_coverage_configs:
    print(coverage.name)
    coverage.infer_roi_coverage()
    coverage.infer_roiless_indices(sub_indices)
    

predefined
mean_euc_nearest_voxels
mean_cos_nearest_voxels
ms_euc_nearest_voxels
ms_cos_nearest_voxels
ms_euc_nearest_center
ms_cos_nearest_center


In [21]:
tenzor_dir = '/home/matanyaw/data/roi_coverages_tenzors/'

for coverage in roi_coverage_configs:
        coverage.save_into_tezor(save_path=tenzor_dir)

In [ ]:
predefined_config.save_into_tezor(save_path=tenzor_dir)

tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 1, 1, 1],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]], dtype=torch.int8)

In [ ]:
# now let's load the tenzor
tenzor = torch.load(tenzor_dir + predefined_config.name + '.pt')
tenzor

tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 1, 1, 1],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]], dtype=torch.int8)

In [20]:
roi_names = ROI_coverage.get_roi_names(subject=1)
row = roi_names.index('FFA-1')
row

ROI 'mTL-bodies' not found for subject 1
ROI 'mTL-faces' not found for subject 1
ROI 'aTL-faces' not found for subject 1
ROI 'mTL-words' not found for subject 1


4

In [ ]:
tenzor.shape
for roi_idx, roi_name in enumerate(ROI_names):
    print(tenzor[roi_idx, :].sum())


tensor(6237)
tensor(780)
tensor(856)
tensor(737)
tensor(882)
tensor(629)
tensor(4669)
tensor(2202)
tensor(1061)
tensor(907)
tensor(1778)
tensor(892)
tensor(490)
tensor(1154)
tensor(1819)
tensor(1519)
tensor(1417)
tensor(1249)
tensor(1204)
tensor(1296)


In [ ]:
num_voxels = voxel_embeddings.shape[0]
num_rois = len(ROI_names)
roi_tensor = torch.zeros((num_rois, num_voxels), dtype=torch.int8)

roi_tensor[0, 0:1000] = 1
roi_tensor[1, 1000:2000] = 1
roi_tensor[2, 2000:3000] = 1
roi_tensor[3, 3000:4000] = 1
roi_tensor[4, 4000:5000] = 1
roi_tensor[5, 5000:6000] = 1
roi_tensor[6, 6000:7000] = 1
roi_tensor[7, 7000:8000] = 1
roi_tensor[8, 8000:9000] = 1

torch.save(roi_tensor, '/home/matanyaw/data/roi_coverages_tenzors/test_tenzor.pt')
